In [1]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np
import polars as pl


In [2]:
TARGET_ID = "CHEMBL203"
target = new_client.target
activity = new_client.activity
molecule = new_client.molecule

# EFGR activities
print("Fetching efgr activities from chemble database")

results =activity.filter(
    target_chembl_id = TARGET_ID,
    assay_type="B",
    standard_type = 'IC50',
    standard_units = 'nM',
    standard_relation = '=',

).only(
    "molecule_chembl_id",
    "standard_value",
    "standard_type",
    "standard_unit",
    "standard_relation",
    "pchembl_value",
    "assay_chembl_id",
    "target_chembl_id",

)

Fetching efgr activities from chemble database


In [3]:
total_records = len(results)
print(f"Total records found: {total_records}")

Total records found: 18381


In [5]:
from tqdm import tqdm

In [6]:
# collect data with progress
data = []
for records in tqdm(results, total = total_records):
    data.append(records)

# convert to dataframe
df = pl.DataFrame(data)


100%|██████████| 18381/18381 [10:44<00:00, 28.53it/s]  


In [13]:
df.write_parquet("../data/egfr_master_dataset.parquet")

In [8]:
df.head(20)

assay_chembl_id,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_value,target_chembl_id,type,value
str,str,str,str,str,str,str,str,str,str
"""CHEMBL674637""","""CHEMBL68920""","""7.39""","""=""","""=""","""IC50""","""41.0""","""CHEMBL203""","""IC50""","""0.041"""
"""CHEMBL674637""","""CHEMBL69960""","""6.77""","""=""","""=""","""IC50""","""170.0""","""CHEMBL203""","""IC50""","""0.17"""
"""CHEMBL677833""","""CHEMBL137635""","""5.03""","""=""","""=""","""IC50""","""9300.0""","""CHEMBL203""","""IC50""","""9.3"""
"""CHEMBL674643""","""CHEMBL306988""",null,"""=""","""=""","""IC50""","""500000.0""","""CHEMBL203""","""IC50""","""500.0"""
"""CHEMBL674643""","""CHEMBL66879""",null,"""=""","""=""","""IC50""","""3000000.0""","""CHEMBL203""","""IC50""","""3000.0"""
…,…,…,…,…,…,…,…,…,…
"""CHEMBL674643""","""CHEMBL421877""",null,"""=""","""=""","""IC50""","""850000.0""","""CHEMBL203""","""IC50""","""850.0"""
"""CHEMBL674643""","""CHEMBL310798""","""5.52""","""=""","""=""","""IC50""","""3000.0""","""CHEMBL203""","""IC50""","""3.0"""
"""CHEMBL677833""","""CHEMBL135592""","""5.10""","""=""","""=""","""IC50""","""8000.0""","""CHEMBL203""","""IC50""","""8.0"""


In [3]:
import os
print(os.getcwd())

/Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks


In [4]:
df = pd. read_parquet('../data/egfr_master_dataset.parquet')

In [5]:
df.shape

(18381, 10)

In [6]:
df.head()

,assay_chembl_id,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_value,target_chembl_id,type,value
0,CHEMBL674637,CHEMBL68920,7.39,=,=,IC50,41.0,CHEMBL203,IC50,0.041
1,CHEMBL674637,CHEMBL69960,6.77,=,=,IC50,170.0,CHEMBL203,IC50,0.17
2,CHEMBL677833,CHEMBL137635,5.03,=,=,IC50,9300.0,CHEMBL203,IC50,9.3
3,CHEMBL674643,CHEMBL306988,None,=,=,IC50,500000.0,CHEMBL203,IC50,500.0
4,CHEMBL674643,CHEMBL66879,None,=,=,IC50,3000000.0,CHEMBL203,IC50,3000.0


In [33]:
unique_count = df['assay_chembl_id'].nunique()

In [34]:
unique_count

1910

In [19]:
df.columns

Index(['assay_chembl_id', 'molecule_chembl_id', 'pchembl_value', 'relation',
       'standard_relation', 'standard_type', 'standard_value',
       'target_chembl_id', 'type', 'value'],
      dtype='object')

In [ ]:
Usage in your notebook:

  fetcher = ChEMBLFetcher("CHEMBL203")
  raw = fetcher.fetch_raw()
  curated = fetcher.curate(raw)

  # Check flagged rows
  flagged = curated[~curated["pchembl_agreement"]]
  print(f"{len(flagged)} compounds with pchembl disagreement — review these")

In [12]:
os.chdir('../')

In [13]:
os.getcwd()

'/Users/ankitkumar/Documents/Projects/egfr-pic50-prediction'

In [14]:
from src.components.data_loader import ChEMBLFetcher

In [28]:
len(df)

18381

In [29]:
len(chembl_ids)

10524

In [24]:
from chembl_webresource_client.new_client import new_client
from tqdm.auto import tqdm
import pandas as pd

molecule = new_client.molecule

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

chembl_ids = (
    df["molecule_chembl_id"]
    .dropna()
    .unique()
    .tolist()
)

batch_size = 500
all_molecules = []

batches = list(chunks(chembl_ids, batch_size))

for batch in tqdm(
    batches,
    desc="Fetching molecules",
    unit="batch"
):
    mols = molecule.filter(
        molecule_chembl_id__in=batch
    ).only(
        [
            "molecule_chembl_id",
            "molecule_structures"
        ]
    )

    all_molecules.extend(list(mols))

Fetching molecules:   0%|          | 0/22 [00:00<?, ?batch/s]

In [25]:
mol_records = []

for mol in tqdm(
    all_molecules,
    desc="Extracting SMILES",
    unit="molecule"
):
    structures = mol.get("molecule_structures")

    mol_records.append({
        "molecule_chembl_id": mol["molecule_chembl_id"],
        "canonical_smiles": (
            structures.get("canonical_smiles")
            if structures else None
        )
    })

df_mol = pd.DataFrame(mol_records)

Extracting SMILES:   0%|          | 0/10524 [00:00<?, ?molecule/s]

In [27]:
len(df_mol)

10524